### Goal: Write a clean Python script using httpx or requests and pydantic to fetch JSON data from a public API, validate schema, and structure into a DataFrame.

In [2]:
import httpx
response = httpx.get("https://www.ncpedia.org/south-mountains-state-park")

In [3]:
response.status_code

200

In [4]:
response.text

'\n\n<!-- THEME DEBUG -->\n<!-- THEME HOOK: \'html\' -->\n<!-- FILE NAME SUGGESTIONS:\n   ▪️ html--node--53437.html.twig\n   ▪️ html--node--%.html.twig\n   ▪️ html--node.html.twig\n   ✅ html.html.twig\n-->\n<!-- 💡 BEGIN CUSTOM TEMPLATE OUTPUT from \'themes/custom/nmc_drupalbase/templates/layout/html.html.twig\' -->\n<!DOCTYPE html>\n<html lang="en" dir="ltr" prefix="og: https://ogp.me/ns#">\n\t<head>\n\t\t<meta charset="utf-8" />\n<noscript><style>form.antibot > * { display: none !important; } form.antibot noscript { display: block !important; }</style>\n</noscript><meta name="description" content="Established: 1978GPS Coordinates: 35.5963,-81.6Size: 18,627 acres See also: Exploring North Carolina: North Carolina State Parks, Trails, Lakes" />\n<link rel="canonical" href="https://www.ncpedia.org/south-mountains-state-park" />\n<meta name="Generator" content="Drupal 11 (https://www.drupal.org)" />\n<meta name="MobileOptimized" content="width" />\n<meta name="HandheldFriendly" content="t

In [1]:
import httpx

# USGS OGC API endpoint for continuous water data items
url = "https://api.waterdata.usgs.gov/ogcapi/v1/collections/continuous/items"

# Map the frontend query parameters to the API parameters
params = {
    "monitoring_location_id": "USGS-02143000",
    "parameter_code": "00065",  # 00065 corresponds to gage height
    "datetime": "P7D",          # ISO 8601 duration for the past 7 days
    "f": "json"                 # Request response format as JSON
}

headers = {
    "Accept": "application/json"
}


In [2]:
response_usgs = httpx.get(url)
print(response_usgs.text)

{
  "type": "FeatureCollection",
  "features": [
    {
      "type": "Feature",
      "properties": {
        "time_series_id": "00009b3da7e44a96a283bc413ff2de48",
        "monitoring_location_id": "USGS-10172640",
        "parameter_code": "00060",
        "statistic_id": "00011",
        "time": "2025-10-22T15:00:00+00:00",
        "value": "74.7",
        "unit_of_measure": "ft^3/s",
        "approval_status": "Approved",
        "qualifier": null,
        "last_modified": "2026-03-16T17:47:57.805796+00:00",
        "method_category": null
      },
      "id": "b55ccd0f-f41a-4817-bf59-c105b15e0779",
      "geometry": {
        "type": "Point",
        "coordinates": [
          -112.13859444444445,
          40.780852777777774
        ]
      }
    },
    {
      "type": "Feature",
      "properties": {
        "time_series_id": "00009b3da7e44a96a283bc413ff2de48",
        "monitoring_location_id": "USGS-10172640",
        "parameter_code": "00060",
        "statistic_id": "00011",
 

In [3]:
response_usgs.url

URL('https://api.waterdata.usgs.gov/ogcapi/v1/collections/continuous/items')

In [25]:
from datetime import datetime, timedelta, timezone
import httpx
import json

# USGS OGC API endpoint for continuous water data items
url = "https://api.waterdata.usgs.gov/ogcapi/v1/collections/continuous/items"

# Dynamically calculate the past 7 days in ISO 8601 format (Start/End)
end_time = datetime.now(timezone.utc)
start_time = end_time - timedelta(days=7)

# Format into an explicit interval string: "YYYY-MM-DDTHH:MM:SSZ/YYYY-MM-DDTHH:MM:SSZ"
datetime_interval = f"{start_time.strftime('%Y-%m-%dT%H:%M:%SZ')}/{end_time.strftime('%Y-%m-%dT%H:%M:%SZ')}"

params = {
    "monitoring_location_id": "USGS-02143000",
    "parameter_code": "00065",  # Gage height
    "datetime": datetime_interval,
    "f": "json"
}

headers = {
    "Accept": "application/json"
}

def fetch_usgs_data():
    with httpx.Client() as client:
        try:
            response = client.get(url, params=params, headers=headers, timeout=30.0)
            response.raise_for_status()
            return response.json()
            
        except httpx.HTTPStatusError as e:
            print(f"HTTP error occurred: {e.response.status_code} - {e.response.text}")
        except httpx.RequestError as e:
            print(f"An error occurred while requesting {e.request.url!r}: {e}")

if __name__ == "__main__":
    water_data = fetch_usgs_data()
    if water_data:
        print(f"Successfully retrieved {len(water_data.get('features', []))} records.")
        print(json.dumps(water_data, indent=2)[:500])

Successfully retrieved 10 records.
{
  "type": "FeatureCollection",
  "features": [
    {
      "type": "Feature",
      "properties": {
        "time_series_id": "9db37b4cf4e1405c8e0bda80602da55e",
        "monitoring_location_id": "USGS-02143000",
        "parameter_code": "00065",
        "statistic_id": "00011",
        "time": "2026-10-12T21:15:00+00:00",
        "value": "2.36",
        "unit_of_measure": "ft",
        "approval_status": "Provisional",
        "qualifier": null,
        "last_modified": "2026-10-12T21:40:17


In [4]:
from datetime import datetime, timedelta, timezone
import httpx
import pandas as pd

# 1. Setup Request Parameters
url = "https://api.waterdata.usgs.gov/ogcapi/v1/collections/continuous/items"

end_time = datetime.now(timezone.utc)
start_time = end_time - timedelta(days=7)
datetime_interval = f"{start_time.strftime('%Y-%m-%dT%H:%M:%SZ')}/{end_time.strftime('%Y-%m-%dT%H:%M:%SZ')}"

params = {
    "monitoring_location_id": "USGS-02143000",
    "parameter_code": "00065",  # Gage height
    "datetime": datetime_interval,
    "f": "json"
}

headers = {"Accept": "application/json"}

# 2. Fetch Data
with httpx.Client() as client:
    response = client.get(url, params=params, headers=headers, timeout=30.0)
    response.raise_for_status()
    raw_data = response.json()

# 3. Convert JSON to Pandas DataFrame
# GeoJSON stores individual rows inside the 'features' list. 
# Inside each feature, the actual data metrics sit under properties.
features = raw_data.get("features", [])

if features:
    # pd.json_normalize flattens the nested dictionary structure 
    df = pd.json_normalize(features)
    
    # Optional: Clean up column names by dropping the 'properties.' prefix 
    # that json_normalize automatically adds to GeoJSON keys
    df.columns = df.columns.str.replace("properties.", "", regex=False)
    
    # Convert time column to actual datetime objects for easier plotting/filtering
    if "time" in df.columns:
        df["time"] = pd.to_datetime(df["time"])
        
    # Convert water measurement values to numeric (USGS API transmits them as strings to preserve precision)
    if "value" in df.columns:
        df["value"] = pd.to_numeric(df["value"], errors="coerce")

    print(f"Successfully loaded {len(df)} rows into a DataFrame!")
    print(df[["time", "value", "unit_of_measure", "approval_status"]].head())
else:
    print("No data features found in response.")

Successfully loaded 10 rows into a DataFrame!
                       time  value unit_of_measure approval_status
0 2026-10-15 15:15:00+00:00   1.56              ft     Provisional
1 2026-10-15 15:30:00+00:00   1.56              ft     Provisional
2 2026-10-15 15:45:00+00:00   1.56              ft     Provisional
3 2026-10-15 16:00:00+00:00   1.56              ft     Provisional
4 2026-10-15 16:15:00+00:00   1.56              ft     Provisional


In [6]:
features

[{'type': 'Feature',
  'properties': {'time_series_id': '9db37b4cf4e1405c8e0bda80602da55e',
   'monitoring_location_id': 'USGS-02143000',
   'parameter_code': '00065',
   'statistic_id': '00011',
   'time': '2026-10-15T15:15:00+00:00',
   'value': '1.56',
   'unit_of_measure': 'ft',
   'approval_status': 'Provisional',
   'qualifier': None,
   'last_modified': '2026-10-15T15:42:20.025412+00:00',
   'method_category': None},
  'id': '49b1f71c-8006-469f-8dbe-904428393fb8',
  'geometry': {'type': 'Point',
   'coordinates': [-81.4033333333333, 35.6844444444444]}},
 {'type': 'Feature',
  'properties': {'time_series_id': '9db37b4cf4e1405c8e0bda80602da55e',
   'monitoring_location_id': 'USGS-02143000',
   'parameter_code': '00065',
   'statistic_id': '00011',
   'time': '2026-10-15T15:30:00+00:00',
   'value': '1.56',
   'unit_of_measure': 'ft',
   'approval_status': 'Provisional',
   'qualifier': None,
   'last_modified': '2026-10-15T15:42:20.025412+00:00',
   'method_category': None},
  'id

In [7]:
df2 = pd.json_normalize(features)
df2

,type,id,properties.time_series_id,properties.monitoring_location_id,properties.parameter_code,properties.statistic_id,properties.time,properties.value,properties.unit_of_measure,properties.approval_status,properties.qualifier,properties.last_modified,properties.method_category,geometry.type,geometry.coordinates
0,Feature,49b1f71c-8006-469f-8dbe-904428393fb8,9db37b4cf4e1405c8e0bda80602da55e,USGS-02143000,00065,00011,2026-10-15T15:15:00+00:00,1.56,ft,Provisional,None,2026-10-15T15:42:20.025412+00:00,None,Point,"[-81.4033333333333, 35.6844444444444]"
1,Feature,7c9a7941-51c0-4ed6-a650-393d45a0e50a,9db37b4cf4e1405c8e0bda80602da55e,USGS-02143000,00065,00011,2026-10-15T15:30:00+00:00,1.56,ft,Provisional,None,2026-10-15T15:42:20.025412+00:00,None,Point,"[-81.4033333333333, 35.6844444444444]"
2,Feature,5f0228b1-3b4c-4be9-88bd-8a7ca0a0af49,9db37b4cf4e1405c8e0bda80602da55e,USGS-02143000,00065,00011,2026-10-15T15:45:00+00:00,1.56,ft,Provisional,None,2026-10-15T16:40:19.570628+00:00,None,Point,"[-81.4033333333333, 35.6844444444444]"
3,Feature,ca06c2f7-28db-43bb-a1a0-b700413dcf84,9db37b4cf4e1405c8e0bda80602da55e,USGS-02143000,00065,00011,2026-10-15T16:00:00+00:00,1.56,ft,Provisional,None,2026-10-15T16:40:19.570628+00:00,None,Point,"[-81.4033333333333, 35.6844444444444]"
4,Feature,e3a6e3ce-e24c-460a-896e-45d8957d69de,9db37b4cf4e1405c8e0bda80602da55e,USGS-02143000,00065,00011,2026-10-15T16:15:00+00:00,1.56,ft,Provisional,None,2026-10-15T16:40:19.570628+00:00,None,Point,"[-81.4033333333333, 35.6844444444444]"
5,Feature,2a9341ec-291b-489d-85a8-dca851046f67,9db37b4cf4e1405c8e0bda80602da55e,USGS-02143000,00065,00011,2026-10-15T16:30:00+00:00,1.56,ft,Provisional,None,2026-10-15T16:40:19.570628+00:00,None,Point,"[-81.4033333333333, 35.6844444444444]"
6,Feature,9c9a1b03-9517-4f94-8293-b6681834fd1e,9db37b4cf4e1405c8e0bda80602da55e,USGS-02143000,00065,00011,2026-10-15T16:45:00+00:00,1.56,ft,Provisional,None,2026-10-15T17:41:23.062902+00:00,None,Point,"[-81.4033333333333, 35.6844444444444]"
7,Feature,2f8fb857-b279-458e-a5b0-496d491f6abd,9db37b4cf4e1405c8e0bda80602da55e,USGS-02143000,00065,00011,2026-10-15T17:00:00+00:00,1.56,ft,Provisional,None,2026-10-15T17:41:23.062902+00:00,None,Point,"[-81.4033333333333, 35.6844444444444]"
8,Feature,b9242056-6933-458b-af3c-cc86132d225a,9db37b4cf4e1405c8e0bda80602da55e,USGS-02143000,00065,00011,2026-10-15T17:15:00+00:00,1.56,ft,Provisional,None,2026-10-15T17:41:23.062902+00:00,None,Point,"[-81.4033333333333, 35.6844444444444]"
9,Feature,4cc505a8-c927-4b09-96a3-948be2a7e0cf,9db37b4cf4e1405c8e0bda80602da55e,USGS-02143000,00065,00011,2026-10-15T17:30:00+00:00,1.55,ft,Provisional,None,2026-10-15T17:41:23.062902+00:00,None,Point,"[-81.4033333333333, 35.6844444444444]"
